# BRO coding server - Qwen2.5-Coder 7B (free, on Colab)

Run the cells **in order** (Runtime > Run all also works). At the end you get one line like
`https://something.trycloudflare.com|SECRET` - paste it into **BRO > Settings > More free AI keys > Colab Qwen**.

Before you start: **Runtime > Change runtime type > T4 GPU**. Keep this browser tab open while you code with BRO.
Each new session gives a NEW line, so paste the new one into BRO.

## 1. Check the GPU

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(out.stdout.strip() or "NO GPU FOUND")
assert out.stdout.strip(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run this cell again."

## 2. Get the model
Looks for the file in your Google Drive first. If it is not there, it downloads the official file from Hugging Face (about 6.25 GB, a few minutes).
Change `MODEL_FILE` if you chose another size (for example `qwen2.5-coder-7b-instruct-q8_0.gguf`).

In [ ]:
import glob, os, shutil

MODEL_FILE = "qwen2.5-coder-7b-instruct-q6_k.gguf"
MODEL_REPO = "Qwen/Qwen2.5-Coder-7B-Instruct-GGUF"
LOCAL = "/content/model.gguf"

found = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    hits = glob.glob("/content/drive/MyDrive/**/" + MODEL_FILE, recursive=True)
    found = hits[0] if hits else None
except Exception as e:
    print("Drive not used:", e)

if found:
    print("Found in Drive:", found)
    shutil.copyfile(found, LOCAL)
else:
    print("Not in Drive - downloading from Hugging Face ...")
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
    shutil.copyfile(path, LOCAL)

print("Model ready: %.2f GB" % (os.path.getsize(LOCAL) / 1e9))

## 3. Install the model server (with GPU support)
First tries a ready-made GPU build (fast). If that does not give GPU support it builds one from source, which can take 10-20 minutes - that is normal.

In [ ]:
import subprocess, sys

def pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)

def gpu_ok():
    r = subprocess.run([sys.executable, "-c",
        "import llama_cpp; print(llama_cpp.llama_supports_gpu_offload())"], capture_output=True, text=True)
    return r.stdout.strip().endswith("True")

ok = False
for tag in ("cu124", "cu123", "cu122", "cu121"):
    pip("llama-cpp-python[server]", "--force-reinstall", "--no-cache-dir",
        "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/" + tag)
    if gpu_ok():
        print("GPU build installed from the", tag, "wheel")
        ok = True
        break

if not ok:
    print("No ready-made GPU build worked - building from source (10-20 min) ...")
    import os
    env = dict(os.environ, CMAKE_ARGS="-DGGML_CUDA=on", FORCE_CMAKE="1")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python[server]",
                        "--force-reinstall", "--no-cache-dir"], env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:])
    ok = gpu_ok()

assert ok, "Could not get GPU support. Copy the red/grey text above and send it to Claude."
print("Server software ready.")

## 4. Start the model server

In [ ]:
import secrets, subprocess, sys, time, urllib.request, json

API_KEY = secrets.token_urlsafe(18)
ALIAS = "qwen2.5-coder-7b"

server_log = open("/content/server.log", "w")
server = subprocess.Popen([
    sys.executable, "-m", "llama_cpp.server",
    "--model", "/content/model.gguf", "--model_alias", ALIAS,
    "--n_gpu_layers", "-1", "--n_ctx", "16384",
    "--host", "127.0.0.1", "--port", "8000", "--api_key", API_KEY,
], stdout=server_log, stderr=subprocess.STDOUT)

def ask(path, data=None, timeout=10):
    req = urllib.request.Request("http://127.0.0.1:8000" + path,
        data=None if data is None else json.dumps(data).encode(),
        headers={"Authorization": "Bearer " + API_KEY, "Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=timeout).read())

for i in range(120):
    try:
        ask("/v1/models")
        print("Server is up after", i * 2, "seconds")
        break
    except Exception:
        if server.poll() is not None:
            print(open("/content/server.log").read()[-3000:])
            raise RuntimeError("The model server stopped. Send the text above to Claude.")
        time.sleep(2)
else:
    print(open("/content/server.log").read()[-3000:])
    raise RuntimeError("The model server did not start in 4 minutes.")

t = time.time()
r = ask("/v1/chat/completions", {"model": ALIAS, "max_tokens": 60,
    "messages": [{"role": "user", "content": "Write a Kotlin function that adds two numbers."}]}, timeout=120)
print("Test answer in %.1fs:" % (time.time() - t))
print(r["choices"][0]["message"]["content"])

## 5. Open the tunnel and get the line for BRO

In [ ]:
import os, re, subprocess, time, urllib.request

if not os.path.exists("/usr/local/bin/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/usr/local/bin/cloudflared")
    os.chmod("/usr/local/bin/cloudflared", 0o755)

tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--no-autoupdate",
                           "--url", "http://127.0.0.1:8000"], stdout=tunnel_log, stderr=subprocess.STDOUT)

URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        URL = m.group(0)
        break
    time.sleep(1)
assert URL, "No tunnel address. Run this cell again."

print("=" * 60)
print("PASTE THIS WHOLE LINE INTO BRO (Settings > Colab Qwen):")
print()
print(URL + "|" + API_KEY)
print()
print("=" * 60)
print("Keep this tab open. Run the next cell to keep the session alive.")

## 6. Keep it running (leave this cell running while you use BRO)

In [ ]:
import time
while True:
    if server.poll() is not None:
        print("Model server stopped. Run cell 4 and 5 again.")
        break
    if tunnel.poll() is not None:
        print("Tunnel stopped. Run cell 5 again.")
        break
    time.sleep(30)